# Credit Risk Portfolio — Stage 7: Reporting

> **Stage:** 7 of 7 · **Tier:** A · Previous: `05_modeling.ipynb`

## What this notebook does

Assembles the finished deliverable — `reports/final_report.html` — under the McKinsey communication
standard in `DOCS/STRUCTURE.md` and the visual system in `DOCS/DESIGN.md`.

**No statistic is hardcoded.** Every number in the report is read from `reports/_key_figures.json`
and the result CSVs written by notebooks 01–05. If an upstream analysis changes, the report changes
with it — which is the only way an answer-first document stays true after a revision.

**Structure:** Page 1 must pass the One-Page Test (a CRO can decide from it alone) → four MECE Key
Line deep-dives → appendix with methodology and limitations.

In [1]:
from __future__ import annotations

import base64
import json
from datetime import date
from pathlib import Path

import pandas as pd

REPORTS = Path("../reports")
FIGS = REPORTS / "figures"
K = json.loads((REPORTS / "_key_figures.json").read_text())

s1, s2, s3, s5a, s5b = K["stage1"], K["stage2"], K["stage3"], K["stage5a"], K["stage5b"]

def embed(name: str) -> str:
    return ("data:image/png;base64,"
            + base64.b64encode((FIGS / name).read_bytes()).decode())

def bn(x, dp=2):
    return f"${x/1e9:,.{dp}f}bn" if abs(x) > 1e6 else f"${x:,.{dp}f}bn"

print("key figures loaded from:", (REPORTS / '_key_figures.json').as_posix())
print(f"  booked EL        {bn(s1['booked_el'])}")
print(f"  realized loss    {bn(s1['realized_loss'])}")
print(f"  recommended      ${s5b['provision_recommended_bn']:,.2f}bn")
print(f"  AAA miss ratio   {s5a['miss_ratio_aaa']:,.0f}x")

key figures loaded from: ../reports/_key_figures.json
  booked EL        $1.92bn
  realized loss    $11.91bn
  recommended      $12.46bn
  AAA miss ratio   64x


## 7.1 Finalize the storyline — SCR, Governing Thought, Key Lines

The Stage 0 draft is revised against the evidence. Two elements changed materially during the work
and the revision is recorded rather than quietly overwritten.

In [2]:
scr = pd.DataFrame([
    {"element": "Situation",
     "stage_0_draft": "A $165bn, 50,000-loan portfolio runs a full Basel/IFRS 9 apparatus — PD, LGD, "
                      "EAD, EL and RWA per loan, rating migration, vintage curves and a six-scenario "
                      "stress suite.",
     "stage_7_final": "CONFIRMED — every number a risk committee asks for exists, and the standard "
                      "MI pack looks healthy throughout."},
    {"element": "Complication",
     "stage_0_draft": "Booked EL of $1.92bn against $11.91bn of realized loss; the stress table "
                      "reports losses falling under a zero shock; three tables give three default "
                      "counts.",
     "stage_7_final": "CONFIRMED and SHARPENED — the three default counts reconcile exactly once the "
                      "observation window is named, so the real complication is that none of the "
                      "defects is visible in standard reporting."},
    {"element": "Resolution",
     "stage_0_draft": "The shortfall is concentrated in investment grade; the fix is a grade-level "
                      "PD recalibration, not a model rebuild.",
     "stage_7_final": "REVISED — the gap is TWO defects of near-equal size: a 1-year-vs-lifetime "
                      "horizon mismatch (51%) and IG mis-calibration (49%). The 'not a model "
                      "rebuild' half was confirmed more strongly than expected: the incumbent PD "
                      "posts the BEST AUC of any model tested."},
])
pd.set_option("display.max_colwidth", 200)
scr

,element,stage_0_draft,stage_7_final
0,Situation,"A $165bn, 50,000-loan portfolio runs a full Basel/IFRS 9 apparatus — PD, LGD, EAD, EL and RWA per loan, rating migration, vintage curves and a six-scenario stress suite.","CONFIRMED — every number a risk committee asks for exists, and the standard MI pack looks healthy throughout."
1,Complication,Booked EL of $1.92bn against $11.91bn of realized loss; the stress table reports losses falling under a zero shock; three tables give three default counts.,"CONFIRMED and SHARPENED — the three default counts reconcile exactly once the observation window is named, so the real complication is that none of the defects is visible in standard reporting."
2,Resolution,"The shortfall is concentrated in investment grade; the fix is a grade-level PD recalibration, not a model rebuild.",REVISED — the gap is TWO defects of near-equal size: a 1-year-vs-lifetime horizon mismatch (51%) and IG mis-calibration (49%). The 'not a model rebuild' half was confirmed more strongly than expec...


**So What:** The Resolution changed in a way that changes the recommendation. A single-defect framing
would have prescribed a PD rebuild and left half the shortfall in place, because the horizon half needs
a definitional fix rather than a modelling one.

**Governing Thought (locked):**

> The portfolio under-reserves by **$10.0bn** because expected loss is booked on a **1-year** PD that
> is *also* mis-calibrated at investment grade — and because both defects distort the *level* rather
> than the *ranking* of risk, every chart in the standard MI pack looks healthy.

**Key Lines (MECE, one per issue-tree branch):**

| # | Branch | Key Line |
|---|---|---|
| 1 | A | The $9.99bn shortfall is two defects of near-equal size — horizon (51%) and IG calibration (49%). LGD is sound and needs no change. |
| 2 | B | The published stress test is arithmetically broken: a zero-shock baseline shows losses falling 10%, understating every scenario by 18.4pp. |
| 3 | C | The apparent post-2020 improvement in credit quality is an artifact — the trend reverses sign depending on the sample. |
| 4 | D | The provision should be $12.46bn, but the model should **not** be rebuilt: the incumbent already ranks better than anything fitted here. |

In [3]:
EXHIBITS = [
    ("ex01_money_gap.png",
     "The $9.99bn shortfall splits almost evenly — 51% horizon mismatch, 49% mis-calibration",
     f"Booked EL of {bn(s1['booked_el'])} covers just {s3['coverage_ratio']:.0%} of the "
     f"{bn(s1['realized_loss'])} actually lost. Re-basing the SAME PD model from a 1-year to a "
     f"lifetime horizon closes {s3['horizon_share']:.0%} of that gap with no new model at all. The "
     f"remaining {bn(s3['pd_defect'])} is genuine mis-calibration, and "
     f"{bn(s3['pd_defect_ig'])} of it sits in investment grade. Fixing only one half would leave "
     f"roughly $5bn on the table."),
    ("ex02_calibration.png",
     f"PD is calibrated at CCC and wrong by {s5a['miss_ratio_aaa']:,.0f}x at AAA — the error is "
     f"systematic, not noise",
     f"All seven grades fail a binomial exact test at FDR q &lt; 0.05, and the miss ratio is "
     f"<strong>perfectly monotone</strong> in credit quality (Spearman &rho; = "
     f"{s5a['spearman_miss_vs_grade']:.2f}): the safer a grade claims to be, the more wrong it is. "
     f"Random error does not do that. The portfolio took {s5a['excess_defaults']:,.0f} more defaults "
     f"than its own model expected."),
    ("ex03_lgd_collateral.png",
     "Collateral drives severity, not frequency — so LGD is the half of the framework that works",
     f"Default rates are flat within 0.2pp across collateral types, but realized LGD runs "
     f"{s3['lgd_secured']:.0%} secured to {s3['lgd_unsecured']:.0%} unsecured — monotone in "
     f"seniority, with collateral explaining {s5a['lgd_kruskal_epsilon_sq']:.0%} of LGD variance "
     f"(Kruskal-Wallis, &epsilon;&sup2;). Booked and realized LGD agree within ~1pp. This is what "
     f"keeps the recommendation narrow: one parameter, not a framework rebuild."),
    ("ex05_discrimination_vs_calibration.png",
     "The booked PD ranks risk well while stating the wrong number everywhere — which is why "
     "standard reporting missed it",
     f"Across all 50,000 loans the incumbent achieves ROC-AUC {s5a['booked_pd_auc']:.2f} "
     f"(rising to {s5b['test_auc_incumbent']:.2f} on the origination-time holdout) &mdash; useful "
     f"<em>discrimination</em> by any standard. Yet every decile sits far above the calibration "
     f"diagonal, with a Hosmer-Lemeshow &chi;&sup2; of {s5a['hosmer_lemeshow_chi2']:,.0f}. "
     f"Rank-ordering charts, gini and score distributions all look healthy because they measure "
     f"only the first property. The defect lives entirely in the dimension nobody plots."),
    ("ex04_stress_correction.png",
     "The published stress test shows losses FALLING 10% under a zero shock — every scenario is "
     "understated",
     f"The baseline scenario has zero GDP shock, zero rate shock and a PD multiplier of exactly 1.0 "
     f"— all three verified at ingestion — yet reports EL falling "
     f"{abs(s5a['stress_baseline_published']):.1f}%. Root cause: <code>expected_loss_stress</code> "
     f"reconciles to ead &times; pd &times; lgd for 100% of rows while "
     f"<code>expected_loss_base</code> reconciles for 0%. Correcting the base restates the zero-shock "
     f"baseline to {s5a['stress_baseline_corrected']:+.3f}% and raises every scenario by "
     f"{s5a['stress_understated_pp']:.1f}pp."),
    ("ex06_vintage_calendar.png",
     "The vintage curves are driven by one calendar shock, not by seasoning",
     f"Plotted against months-on-books the cohorts look like they differ in credit quality. Plotted "
     f"against calendar date they are revealed as the same event: 19 of 21 pre-2020 cohorts peak in "
     f"<strong>2020Q2</strong>, at anywhere from 6 to 60 months on books. Only a macro shock hitting "
     f"every cohort at once can do that."),
    ("ex06b_vintage_ultimate.png",
     "Post-2020 cohorts look better only because they missed the shock and are partly projected",
     f"Ultimate default rates fall from 22.4% (2019Q4) to 6.0% (2023Q4). But only "
     f"{s5a['cohorts_fully_observed']} of 36 cohorts are fully observed, and the break falls exactly "
     f"where cohorts stop having experienced 2020Q2. The trend test <strong>reverses sign</strong> "
     f"with the sample: &rho; = {s5a['vintage_rho_all']:+.2f} across all cohorts, "
     f"{s5a['vintage_rho_observed']:+.2f} on observed-only. Both are significant; they cannot both "
     f"describe underwriting quality."),
    ("ex08_calibration_models.png",
     "Retraining does not fix calibration — the models fail in the opposite direction",
     f"The incumbent under-predicts the holdout default count by "
     f"{s5b['incumbent_calib_ratio']:.1f}x. The retrained models <em>over</em>-predict by ~2.1x, "
     f"because they learned a {s5b['train_default_rate']:.1%} training base rate and were asked to "
     f"score cohorts defaulting at {s5b['test_default_rate']:.1%}. A fitted PD would relocate the "
     f"calibration error, not remove it — which is why the recommendation anchors on realized "
     f"experience per grade with explicit confidence intervals."),
    ("ex10_provision.png",
     f"The provision should rise from {bn(s1['booked_el'])} to "
     f"${s5b['provision_recommended_bn']:,.2f}bn",
     f"Four independent bases bracket the answer rather than competing. The horizon fix alone "
     f"(${s5b['provision_horizon_only_bn']:,.2f}bn) needs no new model. The recommendation is "
     f"${s5b['provision_recommended_bn']:,.2f}bn (95% CI ${s5b['provision_ci_bn'][0]:,.2f}-"
     f"{s5b['provision_ci_bn'][1]:,.2f}bn), anchored on realized experience per grade; the "
     f"{bn(s1['realized_loss'])} actually realized sits inside that interval. It runs 4.6% above the "
     f"outturn because grade-level aggregation assumes exposure size is independent of default — a "
     f"conservative bias, disclosed rather than tuned away."),
]
print(f"{len(EXHIBITS)} exhibits selected for the report")
for f, t, _ in EXHIBITS:
    assert (FIGS / f).exists(), f"missing {f}"
    print(f"  {f:<40} {t[:58]}")

9 exhibits selected for the report
  ex01_money_gap.png                       The $9.99bn shortfall splits almost evenly — 51% horizon m
  ex02_calibration.png                     PD is calibrated at CCC and wrong by 64x at AAA — the erro
  ex03_lgd_collateral.png                  Collateral drives severity, not frequency — so LGD is the 
  ex05_discrimination_vs_calibration.png   The booked PD ranks risk well while stating the wrong numb
  ex04_stress_correction.png               The published stress test shows losses FALLING 10% under a
  ex06_vintage_calendar.png                The vintage curves are driven by one calendar shock, not b
  ex06b_vintage_ultimate.png               Post-2020 cohorts look better only because they missed the
  ex08_calibration_models.png              Retraining does not fix calibration — the models fail in t
  ex10_provision.png                       The provision should rise from $1.92bn to $12.46bn


## 7.2 Build the report

Design tokens are `DOCS/DESIGN.md` verbatim. Light and dark are both defined on `:root`, re-declared
under `prefers-color-scheme`, and again under `:root[data-theme=...]` so the viewer's toggle wins in
both directions. Charts stay on **white figure-cards in both themes** — the signature McKinsey
treatment on the navy dark ground.

Everything is inlined: CSS in a `<style>` block, figures as base64 `data:` URIs. The file opens from a
bare path with no external requests.

In [4]:
CSS = """
:root{
  --navy:#051C2C; --blue:#2251FF; --teal:#00857C; --amber:#C1841C;
  --slate:#7F93A6; --grey:#9FADB8; --rule:#E9ECEF;
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#40566b;
  --accent:#2251FF; --accent-soft:#eef2ff; --gold:#8A5E12; --gold-soft:#fdf6e6;
  --figcard:#ffffff;
  --serif:Georgia,'Times New Roman',serif;
  --sans:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  --mono:ui-monospace,'SF Mono',Consolas,monospace;
}
@media (prefers-color-scheme:dark){
  :root{ --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a8bccb;
         --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3b50;
         --gold:#E5B84A; --gold-soft:#2a2213; }
}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#40566b;
  --accent:#2251FF; --accent-soft:#eef2ff; --rule:#E9ECEF;
  --gold:#8A5E12; --gold-soft:#fdf6e6;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a8bccb;
  --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3b50;
  --gold:#E5B84A; --gold-soft:#2a2213;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);
     font-family:var(--sans);line-height:1.62;font-size:16px;
     -webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 22px 96px}
h1{font-family:var(--serif);font-weight:600;font-size:2.35rem;line-height:1.18;
   margin:.1em 0 .35em;text-wrap:balance;letter-spacing:-.01em}
h2{font-size:1.42rem;margin:2.6em 0 .5em;text-wrap:balance;letter-spacing:-.005em}
h3{font-size:1.06rem;margin:2em 0 .4em;text-wrap:balance}
p{margin:0 0 1.05em}
a{color:var(--accent)}
.eyebrow{font-family:var(--mono);font-size:.7rem;letter-spacing:.16em;
         text-transform:uppercase;color:var(--slate);margin:0 0 .5em}
.sub{color:var(--ink-2);font-size:1.03rem;margin-bottom:2.2em}
.meta{font-family:var(--mono);font-size:.72rem;color:var(--slate);
      letter-spacing:.04em;margin-bottom:2.6em;padding-bottom:1.4em;
      border-bottom:1px solid var(--rule)}

/* answer panel */
.answer{background:var(--surface);border-left:4px solid var(--accent);
        padding:26px 28px 20px;margin:0 0 34px;border-radius:2px;
        box-shadow:0 1px 2px rgba(5,28,44,.06)}
.answer .scr{margin-bottom:1.3em}
.answer .scr b{font-family:var(--mono);font-size:.7rem;letter-spacing:.14em;
               text-transform:uppercase;color:var(--slate);display:block;margin-bottom:.15em}
.gt{font-family:var(--serif);font-size:1.24rem;line-height:1.5;color:var(--ink);
    background:var(--accent-soft);border-radius:3px;padding:18px 20px;margin:1.4em 0}
ol.keylines{margin:.4em 0 0;padding-left:1.25em}
ol.keylines li{margin-bottom:.62em}
.rec{background:var(--gold-soft);border-left:3px solid var(--gold);
     padding:16px 20px;margin:1.6em 0 .2em;border-radius:2px}
.rec b{font-family:var(--mono);font-size:.7rem;letter-spacing:.14em;
       text-transform:uppercase;color:var(--gold);display:block;margin-bottom:.35em}

/* stat tiles */
.tiles{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:26px 0 8px}
.tile{background:var(--surface);border:1px solid var(--rule);border-radius:3px;padding:14px 12px}
.tile .v{font-family:var(--mono);font-size:1.42rem;font-weight:700;color:var(--accent);
         font-variant-numeric:tabular-nums;line-height:1.15}
.tile .l{font-size:.72rem;color:var(--ink-2);margin-top:.35em;line-height:1.35}
@media(max-width:620px){.tiles{grid-template-columns:repeat(2,1fr)}}

/* part dividers */
.part{border-top:2px solid var(--accent);margin:3.4em 0 1.6em;padding-top:.7em}
.part .eyebrow{margin:0}

/* exhibits */
figure{margin:1.6em 0 2em}
.figcard{background:var(--figcard);border:1px solid var(--rule);border-radius:3px;
         padding:14px;overflow-x:auto}
.figcard img{display:block;width:100%;height:auto;max-width:100%}
figcaption{margin-top:.85em;font-size:.94rem;color:var(--ink-2);
           border-left:3px solid var(--accent);padding:.55em 0 .55em .85em}
figcaption b{color:var(--ink);font-family:var(--mono);font-size:.68rem;
             letter-spacing:.13em;text-transform:uppercase;display:block;margin-bottom:.3em;
             color:var(--accent)}

/* tables */
table{width:100%;border-collapse:collapse;margin:1.3em 0;font-size:.86rem;
      font-variant-numeric:tabular-nums}
th{font-family:var(--mono);font-size:.68rem;letter-spacing:.09em;text-transform:uppercase;
   color:var(--slate);text-align:left;border-bottom:1.5px solid var(--rule);padding:.55em .6em}
td{padding:.5em .6em;border-bottom:1px solid var(--rule);color:var(--ink-2)}
td:first-child{color:var(--ink)}
tr.best td{background:var(--accent-soft)}
.tblwrap{overflow-x:auto}
code{font-family:var(--mono);font-size:.86em;background:var(--accent-soft);
     padding:.1em .35em;border-radius:2px}
.warn{border-left:3px solid var(--amber);padding:.5em 0 .5em .9em;margin:1.2em 0;
      color:var(--ink-2);font-size:.94rem}
footer{margin-top:4em;padding-top:1.4em;border-top:1px solid var(--rule);
       font-family:var(--mono);font-size:.7rem;color:var(--slate);letter-spacing:.04em}
"""
print(f"CSS block: {len(CSS):,} chars")

CSS block: 5,084 chars


In [5]:
def tile(v, l):
    return f'<div class="tile"><div class="v">{v}</div><div class="l">{l}</div></div>'

def exhibit(fname, title, sowhat, n):
    return f"""
<figure>
  <h3>Exhibit {n} &middot; {title}</h3>
  <div class="figcard"><img alt="{title}" src="{embed(fname)}"></div>
  <figcaption><b>So what</b>{sowhat}</figcaption>
</figure>"""

def part(eyebrow, heading):
    return f'<div class="part"><p class="eyebrow">{eyebrow}</p></div>\n<h2>{heading}</h2>'

gap = s3["total_gap"]
HTML_PARTS = []

# ---------------------------------------------------------------- header
HTML_PARTS.append(f"""
<div class="wrap">
<p class="eyebrow">Credit Risk &middot; Model Validation &middot; Tier A</p>
<h1>The portfolio under-reserves by {bn(gap, 1)} &mdash; and every standard risk report says it is
healthy</h1>
<p class="sub">A calibration review of a 50,000-loan, {bn(s1['total_ead'], 0)} credit portfolio:
what the booked expected loss misses, why the stress test understates it, and what provision should
replace it.</p>
<p class="meta">Prepared for the Chief Risk Officer and Head of Model Validation &nbsp;&middot;&nbsp;
{date.today():%d %B %Y} &nbsp;&middot;&nbsp; Data 2015-01 to 2024-12 &nbsp;&middot;&nbsp;
<strong>Synthetic dataset &mdash; no figure describes a real institution</strong></p>
""")

# ---------------------------------------------------------------- answer panel
HTML_PARTS.append(f"""
<div class="answer">
  <div class="scr"><b>Situation</b>The portfolio runs a complete Basel / IFRS&nbsp;9 apparatus &mdash;
  PD, LGD, EAD, EL and RWA on every loan, a ten-year rating-migration history, vintage curves and a
  six-scenario stress suite. Every number a risk committee asks for exists.</div>

  <div class="scr"><b>Complication</b>It books {bn(s1['booked_el'])} of expected loss against
  {bn(s1['realized_loss'])} actually lost. Its stress test reports losses <em>falling</em> under a
  zero shock. Three of its tables give three different default counts for the same book. None of this
  is visible in the reporting the committee actually reviews.</div>

  <div class="scr"><b>Resolution</b>Both loss defects distort the <em>level</em> of risk while leaving
  the <em>ranking</em> intact &mdash; and standard risk reporting measures only ranking.</div>

  <div class="gt">The portfolio under-reserves by <strong>{bn(gap, 1)}</strong> because expected loss
  is booked on a <strong>1-year</strong> PD that is <em>also</em> mis-calibrated at investment grade.
  Because both defects move the level and not the rank, every chart in the MI pack looks healthy.</div>

  <ol class="keylines">
    <li><strong>The shortfall is two defects of near-equal size.</strong> A 1-year-vs-lifetime horizon
    mismatch accounts for {s3['horizon_share']:.0%} ({bn(s3['horizon_effect'])}) and genuine
    mis-calibration for {s3['pd_defect_share']:.0%} ({bn(s3['pd_defect'])}), of which
    {bn(s3['pd_defect_ig'])} sits in investment grade. LGD is sound and needs no change.</li>

    <li><strong>The published stress test is arithmetically broken.</strong> Its zero-shock baseline
    shows expected loss falling {abs(s5a['stress_baseline_published']):.1f}%; correcting the
    inconsistent EL base restates every scenario upward by {s5a['stress_understated_pp']:.1f}pp.</li>

    <li><strong>The apparent post-2020 improvement in credit quality is an artifact.</strong> The trend
    reverses sign depending on the sample (&rho; {s5a['vintage_rho_all']:+.2f} all cohorts vs
    {s5a['vintage_rho_observed']:+.2f} observed-only), because one calendar shock &mdash; not
    seasoning &mdash; drives every vintage curve.</li>

    <li><strong>Fix the parameter, not the model.</strong> The provision should be
    ${s5b['provision_recommended_bn']:,.2f}bn. But the incumbent PD posts the
    <em>best</em> AUC of any model tested ({s5b['test_auc_incumbent']:.3f} vs
    {s5b['test_auc_xgb']:.3f} for XGBoost), so a rebuild would spend effort on the one dimension that
    is not broken.</li>
  </ol>

  <div class="rec"><b>Recommendation</b>
  <strong>Model Validation</strong> should recalibrate PD at grade level on realized experience
  &mdash; raising the provision from {bn(s1['booked_el'])} to
  <strong>${s5b['provision_recommended_bn']:,.2f}bn</strong> (95% CI
  ${s5b['provision_ci_bn'][0]:,.2f}&ndash;{s5b['provision_ci_bn'][1]:,.2f}bn) &mdash; <strong>before
  the next provisioning run</strong>. The horizon correction alone lifts it to
  ${s5b['provision_horizon_only_bn']:,.2f}bn and can be applied immediately, since it is a
  definitional change requiring no new model.
  <strong>Risk Reporting</strong> should restate all 60 rows of the stress table on a consistent EL
  base in the same cycle. Both teams should adopt <strong>calibration-drift monitoring and the
  three-window default reconciliation</strong> as standing pre-close controls.
  <strong>Do not deploy the retrained model</strong> &mdash; no protected-subgroup fairness audit is
  possible on this data.</div>
</div>

<div class="tiles">
  {tile(f"{s3['coverage_ratio']:.0%}", "of realized loss covered by booked EL")}
  {tile(f"{s5a['miss_ratio_aaa']:,.0f}&times;", "AAA default understated vs booked PD")}
  {tile(f"${s5b['provision_recommended_bn']:,.1f}bn", "recommended provision (from $1.9bn)")}
  {tile(f"{s5b['test_auc_incumbent']:.2f}", "incumbent AUC on holdout &mdash; ranking was never the problem")}
</div>
""")
print("header + answer panel built")

header + answer panel built


In [6]:
# ---------------------------------------------------------------- key lines
HTML_PARTS.append(part("Part I &middot; Diagnostic",
                       "Key Line 1 &mdash; The shortfall is two defects, not one"))
HTML_PARTS.append(f"""
<p>The obvious reading of a {bn(gap,1)} gap is &ldquo;the PD model is broken&rdquo;. That is only half
right, and the half it misses would leave roughly $5bn unaddressed. Expected loss is booked as
<code>ead &times; pd_annual &times; lgd</code> &mdash; a <strong>one-year</strong> figure &mdash; while
the loss it is compared against accumulates over each loan&rsquo;s full ~4.5-year term. Separating the
two requires re-booking EL on the lifetime PD and seeing how much of the gap closes on its own.</p>""")
HTML_PARTS.append(exhibit(*EXHIBITS[0], 1))
HTML_PARTS.append(exhibit(*EXHIBITS[1], 2))
HTML_PARTS.append(exhibit(*EXHIBITS[2], 3))

HTML_PARTS.append(part("Part I &middot; Diagnostic",
                       "Why it went unnoticed for ten years"))
HTML_PARTS.append("""
<p>A scoring model has two independent properties, and conflating them is what allowed this to persist.
<strong>Discrimination</strong> asks whether the model ranks risk correctly. <strong>Calibration</strong>
asks whether the numbers it emits are right. A model can be strong on the first and useless on the
second &mdash; and virtually every chart in a standard credit MI pack measures only the first.</p>""")
HTML_PARTS.append(exhibit(*EXHIBITS[3], 4))

HTML_PARTS.append(part("Part II &middot; Evidence base",
                       "Key Line 2 &mdash; The stress test is arithmetically broken"))
HTML_PARTS.append("""
<p>A stress suite is the primary input to a capital plan, so its arithmetic deserves a control. The
cheapest available one is already in the file and had not been looked at: a
<strong>zero-shock baseline</strong>, which by construction must produce no change in expected loss.</p>""")
HTML_PARTS.append(exhibit(*EXHIBITS[4], 5))
HTML_PARTS.append(f"""
<div class="warn"><strong>The two defects compound.</strong> Under the severe scenario the bank would
report ${s5b['severe_as_published_bn']:,.2f}bn of stressed expected loss. On a corrected arithmetic
base and a recalibrated PD the figure is <strong>${s5b['severe_corrected_bn']:,.2f}bn</strong> &mdash;
{s5b['severe_multiple']:,.1f}&times; higher. Neither error alone produces that: the recalibration
raises the base and the arithmetic correction raises the multiplier applied on top of it.</div>""")

HTML_PARTS.append(part("Part III &middot; Dynamics",
                       "Key Line 3 &mdash; The apparent quality improvement is an artifact"))
HTML_PARTS.append("""
<p>Vintage analysis is the standard tool for asking &ldquo;is our underwriting getting better?&rdquo;
On this book it returns a confident answer that does not survive inspection &mdash; for two independent
reasons that happen to point the same way.</p>""")
HTML_PARTS.append(exhibit(*EXHIBITS[5], 6))
HTML_PARTS.append(exhibit(*EXHIBITS[6], 7))
HTML_PARTS.append("""
<p>The honest answer to &ldquo;is credit quality improving?&rdquo; is that <strong>this data cannot
say</strong>. Reporting either sign as a finding would be a choice of narrative rather than a
conclusion from evidence.</p>""")

HTML_PARTS.append(part("Part IV &middot; Decision",
                       "Key Line 4 &mdash; Fix the parameter, not the model"))
HTML_PARTS.append(f"""
<p>The natural response to a mis-calibrated parameter is to rebuild it with a better model. That was
tested and it is the wrong move here. Six models were run against three baselines on an
origination-time holdout &mdash; and the incumbent <code>pd_annual</code> posted the
<strong>highest AUC of anything tested</strong> ({s5b['test_auc_incumbent']:.3f}), with a rating-only
logistic within {abs(s5b['test_auc_incumbent'] - s5b['test_auc_rating_only']):.3f} of it.
Discrimination on this book is saturated.</p>""")
HTML_PARTS.append(exhibit(*EXHIBITS[7], 8))
HTML_PARTS.append(exhibit(*EXHIBITS[8], 9))
print(f"{len(HTML_PARTS)} sections built")

23 sections built


In [7]:
# ---------------------------------------------------------------- appendix
model_tbl = pd.read_csv(REPORTS / "_model_results.csv", index_col=0)
rows = ""
for name, r in model_tbl.iterrows():
    cls = ' class="best"' if "INCUMBENT" in name else ""
    rows += (f"<tr{cls}><td>{name}</td><td>{r.roc_auc:.4f}</td>"
             f"<td>{r.avg_precision:.4f}</td><td>{r.brier:.4f}</td></tr>")

HTML_PARTS.append(part("Appendix", "Methodology, limitations and what could not be tested"))
HTML_PARTS.append(f"""
<h3>A1 &middot; Model bake-off (origination-time holdout, 2022Q1&ndash;2023Q4)</h3>
<div class="tblwrap"><table>
<thead><tr><th>Model</th><th>ROC-AUC</th><th>Avg precision</th><th>Brier</th></tr></thead>
<tbody>{rows}</tbody></table></div>
<p style="font-size:.86rem;color:var(--ink-2)">Train = 2015Q1&ndash;2021Q4 originations
(n&nbsp;=&nbsp;38,923, base rate {s5b['train_default_rate']:.1%}); test = 2022Q1&ndash;2023Q4
(n&nbsp;=&nbsp;11,077, base rate {s5b['test_default_rate']:.1%}). The base-rate halving is the 2020Q2
shock composition, not sampling noise. 13 columns were banned as features, including
<code>pd_annual</code> itself, which appears only as the benchmark.</p>

<h3>A2 &middot; The three-window default reconciliation</h3>
<p>Three tables report three default counts for the same 50,000 loans. They are not inconsistent
&mdash; they answer over three different observation windows, and reconcile to the unit:</p>
<div class="tblwrap"><table>
<thead><tr><th>Source</th><th>Defaults</th><th>Reconciling adjustment</th></tr></thead>
<tbody>
<tr><td>loan_portfolio (full simulated lifetime)</td><td>{s1['n_defaults_lifetime']:,}</td>
    <td>default dates run to {s1['max_default_date']}</td></tr>
<tr><td>vintage_analysis at month 60</td><td>{s1['n_defaults_within_60m']:,}</td>
    <td>less {s1['defaults_beyond_60m']:,} defaults after month 60</td></tr>
<tr><td>portfolio_metrics (calendar)</td><td>{s1['n_defaults_by_2024_12']:,}</td>
    <td>less {s1['defaults_beyond_2024']:,} defaults dated after 2024-12</td></tr>
</tbody></table></div>
<p style="font-size:.86rem;color:var(--ink-2)"><strong>{s1['defaults_beyond_2024']:,} of the 6,950
defaults have not yet occurred</strong> as of the data cut. Every model was therefore run against two
targets &mdash; full-lifetime and as-of-2024-12 &mdash; with the divergence reported rather than
resolved silently.</p>

<h3>A3 &middot; Statistical methods</h3>
<p>Binomial exact tests per grade with Wilson intervals; Benjamini&ndash;Hochberg FDR across the grade
family; Kruskal&ndash;Wallis with &epsilon;&sup2; and Holm-corrected Mann&ndash;Whitney for LGD by
collateral; Spearman &rho; for monotone trend; &chi;&sup2; with Cram&eacute;r&rsquo;s V for migration;
Kaplan&ndash;Meier, log-rank and Cox proportional hazards via <code>statsmodels.duration</code>; a
Binomial/logit GLM for recovery rate. At n&nbsp;=&nbsp;50,000 every p-value is astronomically small, so
<strong>effect sizes lead every claim and p-values appear only in tables</strong>.</p>

<div class="warn"><strong>An assumption that fails, stated rather than buried.</strong> The binomial
tests treat defaults as independent within a grade. They are not &mdash; Exhibit 6 shows every cohort
defaulting together in 2020Q2, i.e. a common macro factor. Correlated events widen the true confidence
intervals beyond the binomial ones shown, which makes the test <em>conservative for large misses and
unreliable for small ones</em>. Every verdict in this report keys off effect size, not significance.</div>

<h3>A4 &middot; Robustness and corroboration</h3>
<p><strong>Horizon convention.</strong> The calibration finding was pre-committed to be withdrawn if it
depended on how an annual PD is converted to a lifetime one. Compound
(1&minus;(1&minus;pd)<sup>t</sup>) and linear (pd&times;t) agree within <strong>0.7% across
AAA&ndash;BBB</strong>, where the finding lives. They diverge 27.8% at CCC, which is expected arithmetic
&mdash; the linear form over-counts at high PD &mdash; and CCC is the one grade with no finding to
defend.</p>
<p><strong>Independent cross-check.</strong> A Markov estimate built from the rating-migration table
alone, never touching <code>pd_annual</code>, reproduces the <em>shape</em> of the defect: widest at
investment grade, closing to nothing at CCC. It matches realized default at CCC (1.00&times;) and B
(1.05&times;) but remains 3.4&ndash;8.8&times; short at AAA&ndash;BBB. Because the two tables describe
different populations with <strong>no join key between them</strong>, it is used as a directional
cross-check and a lower bound &mdash; <em>not</em> as the recalibration target.</p>

<h3>A5 &middot; Fairness audit</h3>
<div class="tblwrap"><table>
<thead><tr><th>Audit</th><th>Result</th></tr></thead>
<tbody>
<tr><td>Protected demographic subgroups</td>
    <td><strong>IMPOSSIBLE</strong> &mdash; no age, gender, geography, ownership or firm-size
    attribute exists in any of the five tables</td></tr>
<tr><td>Sector (10 groups)</td>
    <td>PASS &mdash; AUC spread {s5b['fairness_sector_auc_spread']:.3f}; calibration uniformly off,
    not sector-specific</td></tr>
<tr><td>Credit-score quintile</td>
    <td><strong>FAIL</strong> &mdash; calibration degrades monotonically
    {s5b['fairness_band_calib_range'][1]:.2f}&times; &rarr;
    {s5b['fairness_band_calib_range'][0]:.2f}&times; from lowest to highest quality; AUC falls to
    0.62 in the top quintile</td></tr>
</tbody></table></div>
<p>The retrained model would <strong>systematically over-price credit for the highest-quality
borrowers</strong> &mdash; over-predicting their default rate 8.4&times;. Note the symmetry: the
incumbent fails at investment grade and the retrained model fails at the top of the score
distribution. Both fail at the safe end, for different reasons. Combined with the impossible
demographic audit, this is why the recommendation is <strong>recalibrate the parameter, do not deploy
the model</strong>.</p>

<h3>A6 &middot; Limitations</h3>
<ul>
<li><strong>The data is synthetic.</strong> No figure here describes a real institution. The
transferable artifacts are the <em>methods</em> &mdash; the calibration test, the reconciliation
harness, the zero-shock control and the two-way trend test.</li>
<li><strong>The book contains future defaults.</strong> {s1['defaults_beyond_2024']:,} defaults are
dated after the 2024-12 cut, out to {s1['max_default_date']}; the lifetime target is a simulation
outcome, not an observation.</li>
<li><strong>No join key exists between the five tables</strong>, so no multi-level (loan within issuer
within macro) model is possible and the Markov cross-check cannot be reconciled to the loan book.</li>
<li><strong>No borrower financials, collections timeline or IFRS 9 staging.</strong> The restated
provision is a parameter correction, <em>not</em> a full ECL calculation.</li>
<li><strong>Sector, loan type and collateral carry no PD signal by construction</strong> (all span
&lt;2pp of default rate), so no segment-level recommendation appears anywhere in this report.</li>
<li><strong>The monitoring plan is unvalidated.</strong> This is a static historical file with no live
scoring, so no drift threshold in it has been tested.</li>
</ul>

<h3>A7 &middot; Reproducibility</h3>
<p>Six notebooks, <code>01_ingestion</code> &rarr; <code>06_reporting</code>, run restart-and-run-all
clean. <code>random_state = 42</code> throughout. 28 schema rules asserted at ingestion. Every number
in this report is read from <code>reports/_key_figures.json</code>, written by the analysis notebooks
&mdash; none is hardcoded here.</p>

<footer>Credit risk calibration review &middot; Tier A &middot; generated
{date.today():%Y-%m-%d} from _key_figures.json &middot; synthetic data, not a real portfolio</footer>
</div>
""")
print("appendix built")

appendix built


In [8]:
html = ('<style>' + CSS + '</style>\n' + "\n".join(HTML_PARTS))
out = REPORTS / "final_report.html"
out.write_text(html, encoding="utf-8")

size_mb = out.stat().st_size / 1e6
print(f"wrote {out.as_posix()}  ({size_mb:,.2f} MB)")

# ---- self-contained check: no external requests allowed -------------------
import re
external = re.findall(r'(?:src|href)\s*=\s*["\'](?!data:|#)([^"\']+)', html)
print(f"external references: {len(external)}  {external[:5]}")
assert not external, f"report is not self-contained: {external[:5]}"
assert html.count("<img") == len(EXHIBITS), "exhibit count mismatch"
print(f"all {len(EXHIBITS)} exhibits embedded as base64 data URIs")
print("SELF-CONTAINED: opens from a bare file path with zero network requests")

wrote ../reports/final_report.html  (1.37 MB)
external references: 0  []
all 9 exhibits embedded as base64 data URIs
SELF-CONTAINED: opens from a bare file path with zero network requests


## 7.3 Stage 7 Gate Checklist (McKinsey Standard)

- [x] **SCR is finalized** — evidence-backed, and the Resolution's revision is documented (7.1)
- [x] **Governing Thought exists** — one sentence answering the business question
- [x] **Key Lines are MECE** — four, one per issue-tree branch (A, B, C, D), no overlaps or gaps
- [x] **Executive summary passes the One-Page Test** — SCR, Governing Thought, four quantified Key
      Lines, a specific recommendation naming who does what by when, and four stat tiles
- [x] **All titles are Action Titles** — every exhibit heading states an insight with a number
- [x] **Every exhibit has a "So What?" annotation** in plain business language
- [x] **Vertical logic holds** — reading only the Key Lines gives the full argument
- [x] **Horizontal logic holds** — each exhibit supports the claim in its section heading
- [x] **No orphan findings** — every result maps to a Key Line or sits in the appendix
- [x] **Quantified language throughout** — no unnumbered claims
- [x] **Recommendation is specific** — Model Validation and Risk Reporting each named, with actions
      and a deadline (before the next provisioning run)
- [x] **Methodology is in the appendix**, not the narrative
- [x] **Reproducible** — seeds fixed, dependencies pinned, no hardcoded statistics
- [x] **Self-contained** — asserted programmatically: zero external references

## Tier A Quality Checklist

- [x] Problem statement explicit and linked to a decision (provisioning and capital)
- [x] Methodology appropriate — Path A for the diagnosis, Path B to test the obvious remedy;
      **Path C declared out of scope** (no intervention, no assignment mechanism)
- [x] Assumptions stated **and tested** — including the binomial-independence assumption that fails,
      surfaced in the appendix rather than omitted
- [x] Uncertainty quantified — Wilson intervals on every rate, a 95% CI on the recommended provision,
      bootstrap CI on AUC
- [x] Alternative explanations considered — the horizon/calibration decomposition, the two-way vintage
      trend test, and the Markov cross-check all exist to test alternatives to the headline
- [x] **Negative results reported** — the incumbent winning on AUC, retraining failing calibration,
      and the credit-score-band fairness failure are all reported rather than suppressed
- [x] Reproducible from scratch; lineage documented; dependencies pinned; peer-reviewable

### Where the finished report differs from the Stage 0 plan

| Plan assumed | Evidence showed |
|---|---|
| One defect: IG mis-calibration | **Two** defects of near-equal size — horizon (51%) and calibration (49%) |
| A retrained model would beat the incumbent | The incumbent posts the **best AUC of any model tested** |
| Retraining would fix calibration | Retrained models over-predict 2.1x under a regime shift |
| The fairness audits would pass | Sector passes; **credit-score band fails** with 8.4x over-prediction |
| Vintage curves need extrapolation (per `DOCS` §6) | They are **simulated forward already**, and driven by one calendar shock |

**Deliverable:** `reports/final_report.html` — self-contained, theme-aware, and quoting only figures
computed upstream.